In [1]:
import boto3
import urllib3
import requests
import pycountry

import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
bucket = "wbg-geography01"
prefix = "GEP/GEP-Climate-2022/"
region = "us-east-1"
s3client = boto3.client("s3", region_name=region, verify=False)

# Loop through the S3 bucket and get all the file keys
more_results = True
try:
    del token  # noqa
except Exception:
    pass
loops = 0

all_res = []
while more_results:
    print(f"Completed loop: {loops}")
    if loops > 0:
        objects = s3client.list_objects_v2(
            Bucket=bucket,
            ContinuationToken=token,  # noqa
            Prefix=prefix,  # noqa
        )
    else:
        objects = s3client.list_objects_v2(Bucket=bucket)
    more_results = objects["IsTruncated"]
    if more_results:
        token = objects["NextContinuationToken"]
    loops += 1
    for res in objects["Contents"]:
        all_res.append(res)

Completed loop: 0
Completed loop: 1


In [3]:
country_keys = []

for res in all_res:
    path = res['Key']
    try:
        base_folder = path.split("/")[-3]
        if path.endswith("/") & ("GEP" in res['Key']) & (base_folder == "GEP-Climate-2022") & (not path.endswith("_scenario/")):
            country_keys.append(path)
    except:
        pass

In [4]:
country_keys

['GEP/GEP-Climate-2022/ao/',
 'GEP/GEP-Climate-2022/bd/',
 'GEP/GEP-Climate-2022/bf/',
 'GEP/GEP-Climate-2022/bi/',
 'GEP/GEP-Climate-2022/bj/',
 'GEP/GEP-Climate-2022/bw/',
 'GEP/GEP-Climate-2022/cd/',
 'GEP/GEP-Climate-2022/cf/',
 'GEP/GEP-Climate-2022/cg/',
 'GEP/GEP-Climate-2022/ci/',
 'GEP/GEP-Climate-2022/cm/',
 'GEP/GEP-Climate-2022/dj/',
 'GEP/GEP-Climate-2022/er/',
 'GEP/GEP-Climate-2022/et/',
 'GEP/GEP-Climate-2022/fm/',
 'GEP/GEP-Climate-2022/ga/',
 'GEP/GEP-Climate-2022/gh/',
 'GEP/GEP-Climate-2022/gm/',
 'GEP/GEP-Climate-2022/gn/',
 'GEP/GEP-Climate-2022/gq/',
 'GEP/GEP-Climate-2022/gw/',
 'GEP/GEP-Climate-2022/hn/',
 'GEP/GEP-Climate-2022/ht/',
 'GEP/GEP-Climate-2022/ke/',
 'GEP/GEP-Climate-2022/kh/',
 'GEP/GEP-Climate-2022/km/',
 'GEP/GEP-Climate-2022/lr/',
 'GEP/GEP-Climate-2022/ls/',
 'GEP/GEP-Climate-2022/mg/',
 'GEP/GEP-Climate-2022/ml/',
 'GEP/GEP-Climate-2022/mm/',
 'GEP/GEP-Climate-2022/mr/',
 'GEP/GEP-Climate-2022/mw/',
 'GEP/GEP-Climate-2022/mz/',
 'GEP/GEP-Clim

In [6]:
# Define input parameters
params = ["GEP Climate Energy Modeling Parameters", "s3://wbg-geography01/GEP/GEP-Climate-2022/{iso2}/inputs/{iso2}-3-country-inputs.csv"]
specs =['GEP Climate Scenario Specs', 's3://wbg-geography01/GEP/GEP-Climate-2022/{iso2}/inputs/{iso2}-3-specs.xlsx']
res = ['GEP Climate Scenario Results', 's3://wbg-geography01/GEP/GEP-Climate-2022/{iso2}/outputs/{iso2}-3-scenarios-results.zip']
sums = ['GEP Climate Scenario Summaries', 's3://wbg-geography01/GEP/GEP-Climate-2022/{iso2}/outputs/{iso2}-3-scenarios-summaries.zip']
column_defs = ['GEP Climate Column Definitions', 's3://wbg-geography01/GEP/GEP-Climate-2022/GEP V.3 Column Description.xlsx']
all_dfs = []
for country_path in country_keys:
    iso2 = country_path.split("/")[-2]
    country_name = pycountry.countries.get(alpha_2=iso2).name
    link = f"https://energydata.info/dataset/{country_name.lower().replace(' ', '-')}-global-electrification-platform-gep"
    out_params = []    
    for param in [params, specs, res, sums, column_defs]:
        cPath = param[1].format(iso2=iso2)        
        cPrefix = cPath.split(bucket + "/")[-1]
        # Check if file exists
        try:
            s3client.head_object(Bucket=bucket, Key=cPrefix)
        except:
            print(f"File not found: {cPath}")      
        out_params.append([param[0], cPath])
    cur_df = pd.DataFrame(out_params, columns=["Name", "Path"])
    cur_df['Country'] = iso2
    cur_df['Country Name'] = country_name
        # Check if link exists
    response = requests.get(link, verify=False)
    if response.status_code == 404:
        print(link)
        cur_df['EDI_link'] = ""
    else:
        cur_df['EDI_link'] = link
    all_dfs.append(cur_df.copy())

https://energydata.info/dataset/congo,-the-democratic-republic-of-the-global-electrification-platform-gep
https://energydata.info/dataset/côte-d'ivoire-global-electrification-platform-gep
https://energydata.info/dataset/micronesia,-federated-states-of-global-electrification-platform-gep
https://energydata.info/dataset/gambia-global-electrification-platform-gep
https://energydata.info/dataset/eswatini-global-electrification-platform-gep
https://energydata.info/dataset/tanzania,-united-republic-of-global-electrification-platform-gep


In [ ]:
pd.concat(all_dfs).to_csv("GEP-Climate-2025_file_index.csv", index=False)

In [ ]:
import pycountry